# Stage 1a: Open-Vocabulary Object Detection Comparison

Side-by-side comparison of open-vocab detectors on the ZBiotics bottle.
Frames are split into four categories under `data/frames/`:

- `positive_easy/` — product clearly visible, well-lit
- `positive_hard/` — product visible but small, occluded, or off-axis
- `negative_easy/` — frame contains no bottles at all
- `negative_hard/` — frame contains a bottle that is *not* the ZBiotics product

Each detector is in its own cell. Models are loaded, run on all frames, and
unloaded before the next is loaded. Per-detection records flow into
`experiments/stage1/ovod_detections.jsonl` (the input to `SSL_EVAL.ipynb`),
and a per-(model, frame) summary is written to `ovod_summary.json`. A
heatmap of `n_detections` × category × model is written to
`ovod_heatmap.png`.

In [ ]:
# Run once to install everything this notebook needs.
# Uses %pip so installs land in the kernel's environment, not the shell's.
# Re-running this cell is safe: pip skips packages already present.
%pip install --upgrade pip
%pip install --upgrade --quiet \
    torch torchvision pillow "numpy<2" \
    transformers accelerate \
    ultralytics \
    open_clip_torch
    matplotlib
    

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw

from transformers import (
    AutoModelForZeroShotObjectDetection,
    AutoProcessor,
    Owlv2ForObjectDetection,
    Owlv2Processor,
    Sam3Model,
    Sam3Processor,
)
from ultralytics import YOLO, YOLOE
from ultralytics.models.yolo.yoloe import YOLOEVPSegPredictor

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
REPO = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
OUTDIR = REPO / "experiments" / "stage1"
OUTDIR.mkdir(parents=True, exist_ok=True)

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float32  # MPS fp16 still has scattered op bugs

print("device:", DEVICE)
print("outdir:", OUTDIR)


def free_memory():
    """Release tensors and clear the MPS cache between models."""
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()


def draw_boxes(img, boxes, scores, color="lime", width=3, max_boxes=10):
    """Return a copy of img with the top-scoring boxes drawn."""
    out = img.copy()
    if not boxes:
        return out
    draw = ImageDraw.Draw(out)
    ranked = sorted(zip(boxes, scores), key=lambda p: p[1], reverse=True)[:max_boxes]
    for box, score in ranked:
        x0, y0, x1, y1 = (int(v) for v in box)
        draw.rectangle([x0, y0, x1, y1], outline=color, width=width)
        draw.text((x0 + 4, y0 + 4), f"{score:.2f}", fill=color)
    return out


def annotate_and_save(img, boxes, scores, model_name, category, name):
    """Save annotated frame to experiments/stage1/<model_name>/<category>_<name>.png."""
    d = OUTDIR / model_name
    d.mkdir(exist_ok=True)
    draw_boxes(img, boxes, scores).save(d / f"{category}_{name}.png")


# Aggregated per-(model, frame) rows for OVOD's own summary table
results = []


def record(model, lane, label, category, name, top, n, ms, threshold=None):
    """Append one row to the aggregated results table."""
    results.append({
        "model": model,
        "lane": lane,
        "label": label,
        "category": category,
        "frame": name,
        "top_score": float(top) if top is not None else None,
        "n_detections": int(n) if n is not None else 0,
        "ms": float(ms) if ms is not None else None,
        "threshold": threshold,
    })


# Per-detection rows for SSL re-ranking input
detections = []


def record_detections(model, lane, label, category, name, frame_path, boxes, scores, threshold):
    """Append one row per detection to the per-detection table."""
    for box, score in zip(boxes, scores):
        detections.append({
            "model": model,
            "lane": lane,
            "label": label,
            "category": category,
            "frame": name,
            "frame_path": str(frame_path),
            "box": [float(v) for v in box],
            "score": float(score),
            "threshold": threshold,
        })


# Load reference image and prompt metadata
with open(REPO / "data" / "references" / "zbiotics.json") as f:
    meta = json.load(f)

primary_prompt = meta["primary_detection_prompt"]
text_queries = [primary_prompt] + meta["alternate_prompts"]

reference = Image.open(REPO / "data" / "references" / "zbiotics.png").convert("RGB")
ref_w, ref_h = reference.size

# Load frames from 4 category subdirectories
frames_root = (REPO / "data" / "frames").resolve()
CATEGORIES = [
    ("pos", "easy", frames_root / "positive_easy"),
    ("pos", "hard", frames_root / "positive_hard"),
    ("neg", "easy", frames_root / "negative_easy"),
    ("neg", "hard", frames_root / "negative_hard"),
]

frames = []
for label, category, subdir in CATEGORIES:
    if not subdir.exists():
        print(f"warning: {subdir} missing, skipping")
        continue
    for path in sorted(subdir.glob("*.png")):
        img = Image.open(path).convert("RGB")
        frames.append((label, category, path.stem, img, path))

cat_counts = {}
for label, category, _, _, _ in frames:
    cat_counts[(label, category)] = cat_counts.get((label, category), 0) + 1
print("reference", reference.size, "frames", len(frames), "by category:", cat_counts)

device: mps
outdir: /Users/alexwang/Documents/GitHub/storymode/experiments/stage1
reference (1000, 1000) frames 9 by category: {('pos', 'easy'): 2, ('pos', 'hard'): 2, ('neg', 'easy'): 3, ('neg', 'hard'): 2}


In [ ]:
# OWLv2 image-guided
owl_id = "google/owlv2-base-patch16-ensemble"
owl_proc = Owlv2Processor.from_pretrained(owl_id)
owl = Owlv2ForObjectDetection.from_pretrained(owl_id, torch_dtype=DTYPE).to(DEVICE).eval()

threshold = 0.10
for label, category, name, img, path in frames:
    inputs = owl_proc(images=img, query_images=reference, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = owl.image_guided_detection(**inputs)
    ms = (time.perf_counter() - t0) * 1000
    target_sizes = torch.tensor([img.size[::-1]]).to(DEVICE)
    res = owl_proc.post_process_image_guided_detection(
        outputs=out, threshold=threshold, target_sizes=target_sizes
    )[0]
    boxes = res["boxes"].cpu().tolist()
    scores = res["scores"].cpu().tolist()
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "owlv2_image_guided", f"{label}_{category}", name)
    record("owlv2_image_guided", "image", label, category, name, top, len(scores), ms, threshold)
    record_detections("owlv2_image_guided", "image", label, category, name,
                      path, boxes, scores, threshold)
    print("owlv2_image", label, category, name, top, len(scores), f"{ms:.0f}ms")

Loading weights: 100%|██████████| 418/418 [00:00<00:00, 9326.38it/s]


owlv2_image pos easy zbiotics1 0.9999988675117493 263 2875ms
owlv2_image pos easy zbiotics2 0.9999988675117493 240 2562ms
owlv2_image pos hard zbiotics_hard1 0.999998927116394 145 2542ms
owlv2_image pos hard zbiotics_hard2 0.999998927116394 218 2496ms
owlv2_image neg easy zbiotics_easy_negative 0.999998927116394 21 2437ms
owlv2_image neg easy zbiotics_easy_negative1 0.999998927116394 180 2469ms
owlv2_image neg easy zbiotics_easy_negative2 0.9999988675117493 79 2469ms
owlv2_image neg hard zbiotics_hard_negative 0.999998927116394 210 2616ms


In [ ]:
# OWLv2 text-guided (same weights, different prompt path)
threshold = 0.10
for label, category, name, img, path in frames:
    inputs = owl_proc(
        text=[text_queries], images=img, return_tensors="pt",
        padding="max_length", truncation=True,
    ).to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = owl(**inputs)
    ms = (time.perf_counter() - t0) * 1000
    target_sizes = torch.tensor([img.size[::-1]]).to(DEVICE)
    res = owl_proc.post_process_grounded_object_detection(
        outputs=out, threshold=threshold, target_sizes=target_sizes
    )[0]
    boxes = res["boxes"].cpu().tolist()
    scores = res["scores"].cpu().tolist()
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "owlv2_text_guided", f"{label}_{category}", name)
    record("owlv2_text_guided", "text", label, category, name, top, len(scores), ms, threshold)
    record_detections("owlv2_text_guided", "text", label, category, name,
                      path, boxes, scores, threshold)
    print("owlv2_text", label, category, name, top, len(scores), f"{ms:.0f}ms")

del owl, owl_proc
free_memory()

In [ ]:
# YOLO-World v2
yw = YOLO("yolov8s-worldv2.pt")
yw.set_classes(text_queries)

threshold = 0.05
for label, category, name, img, path in frames:
    t0 = time.perf_counter()
    res = yw.predict(source=img, device=DEVICE, conf=threshold, verbose=False)[0]
    ms = (time.perf_counter() - t0) * 1000
    has_boxes = res.boxes is not None and len(res.boxes) > 0
    boxes = res.boxes.xyxy.cpu().tolist() if has_boxes else []
    scores = res.boxes.conf.cpu().tolist() if has_boxes else []
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "yolo_world_v2", f"{label}_{category}", name)
    record("yolo_world_v2", "text", label, category, name, top, len(scores), ms, threshold)
    record_detections("yolo_world_v2", "text", label, category, name,
                      path, boxes, scores, threshold)
    print("yolo_world", label, category, name, top, len(scores), f"{ms:.0f}ms")

del yw
free_memory()

In [ ]:
# YOLOE text-prompt mode
yoloe = YOLOE("yoloe-11s-seg.pt")
yoloe.set_classes(text_queries, yoloe.get_text_pe(text_queries))

threshold = 0.05
for label, category, name, img, path in frames:
    t0 = time.perf_counter()
    res = yoloe.predict(source=img, device=DEVICE, conf=threshold, verbose=False)[0]
    ms = (time.perf_counter() - t0) * 1000
    has_boxes = res.boxes is not None and len(res.boxes) > 0
    boxes = res.boxes.xyxy.cpu().tolist() if has_boxes else []
    scores = res.boxes.conf.cpu().tolist() if has_boxes else []
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "yoloe_text", f"{label}_{category}", name)
    record("yoloe_text", "text", label, category, name, top, len(scores), ms, threshold)
    record_detections("yoloe_text", "text", label, category, name,
                      path, boxes, scores, threshold)
    print("yoloe_text", label, category, name, top, len(scores), f"{ms:.0f}ms")

In [ ]:
# YOLOE visual-prompt mode
# Reference is a clean shot of just the bottle, so the prompt bbox spans the
# whole reference image.
visual_prompts = {
    "bboxes": np.array([[0, 0, ref_w, ref_h]]),
    "cls": np.array([0]),
}

threshold = 0.05
for label, category, name, img, path in frames:
    t0 = time.perf_counter()
    res = yoloe.predict(
        source=img,
        refer_image=reference,
        visual_prompts=visual_prompts,
        predictor=YOLOEVPSegPredictor,
        device=DEVICE,
        conf=threshold,
        verbose=False,
    )[0]
    ms = (time.perf_counter() - t0) * 1000
    has_boxes = res.boxes is not None and len(res.boxes) > 0
    boxes = res.boxes.xyxy.cpu().tolist() if has_boxes else []
    scores = res.boxes.conf.cpu().tolist() if has_boxes else []
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "yoloe_visual", f"{label}_{category}", name)
    record("yoloe_visual", "image", label, category, name, top, len(scores), ms, threshold)
    record_detections("yoloe_visual", "image", label, category, name,
                      path, boxes, scores, threshold)
    print("yoloe_visual", label, category, name, top, len(scores), f"{ms:.0f}ms")

del yoloe
free_memory()

In [ ]:
# SAM 3 text-prompt mode
# Gated repo: requires HF login. Run `from huggingface_hub import login; login()`
# in a cell first if you haven't already. SAM 3 visual-prompt mode is not
# exposed by the transformers integration in this version, so it's skipped.
sam3_id = "facebook/sam3"
sam3_proc = Sam3Processor.from_pretrained(sam3_id)
sam3 = Sam3Model.from_pretrained(sam3_id, torch_dtype=DTYPE).to(DEVICE).eval()


def sam3_postprocess(out, target_size, threshold):
    """Pull boxes and scores from a SAM 3 output, tolerant of API drift."""
    if hasattr(sam3_proc, "post_process_grounded_detection"):
        sizes = torch.tensor([target_size]).to(DEVICE)
        res = sam3_proc.post_process_grounded_detection(
            outputs=out, threshold=threshold, target_sizes=sizes
        )[0]
        return res["boxes"].cpu().tolist(), res["scores"].cpu().tolist()
    boxes = out.pred_boxes[0].cpu().tolist() if hasattr(out, "pred_boxes") else []
    scores = out.scores[0].cpu().tolist() if hasattr(out, "scores") else []
    return boxes, scores


threshold = 0.30
for label, category, name, img, path in frames:
    inputs = sam3_proc(images=img, text=[primary_prompt], return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = sam3(**inputs)
    ms = (time.perf_counter() - t0) * 1000
    boxes, scores = sam3_postprocess(out, img.size[::-1], threshold)
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "sam3_text", f"{label}_{category}", name)
    record("sam3_text", "text", label, category, name, top, len(scores), ms, threshold)
    record_detections("sam3_text", "text", label, category, name,
                      path, boxes, scores, threshold)
    print("sam3_text", label, category, name, top, len(scores), f"{ms:.0f}ms")

del sam3, sam3_proc
free_memory()

In [ ]:
# Grounding DINO — wants lowercase prompts joined by periods
gd_id = "IDEA-Research/grounding-dino-tiny"
gd_proc = AutoProcessor.from_pretrained(gd_id)
gd = AutoModelForZeroShotObjectDetection.from_pretrained(
    gd_id, torch_dtype=DTYPE
).to(DEVICE).eval()

gd_text = " . ".join(p.lower().strip(".") for p in text_queries) + " ."
box_thr, text_thr = 0.25, 0.20

for label, category, name, img, path in frames:
    inputs = gd_proc(images=img, text=gd_text, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = gd(**inputs)
    ms = (time.perf_counter() - t0) * 1000
    res = gd_proc.post_process_grounded_object_detection(
        out,
        inputs.input_ids,
        threshold=box_thr,
        text_threshold=text_thr,
        target_sizes=[img.size[::-1]],
    )[0]
    boxes = res["boxes"].cpu().tolist()
    scores = res["scores"].cpu().tolist()
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "grounding_dino", f"{label}_{category}", name)
    record("grounding_dino_tiny", "text", label, category, name, top, len(scores), ms, box_thr)
    record_detections("grounding_dino_tiny", "text", label, category, name,
                      path, boxes, scores, box_thr)
    print("grounding_dino", label, category, name, top, len(scores), f"{ms:.0f}ms")

del gd, gd_proc
free_memory()

In [ ]:
# OmDet-Turbo — fast text-prompted detector
omdet_id = "omlab/omdet-turbo-swin-tiny-hf"
om_proc = AutoProcessor.from_pretrained(omdet_id)
om = AutoModelForZeroShotObjectDetection.from_pretrained(
    omdet_id, torch_dtype=DTYPE
).to(DEVICE).eval()

threshold = 0.20
for label, category, name, img, path in frames:
    inputs = om_proc(images=img, text=text_queries, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = om(**inputs)
    ms = (time.perf_counter() - t0) * 1000
    res = om_proc.post_process_grounded_object_detection(
        out,
        text_labels=[text_queries],
        threshold=threshold,
        target_sizes=[img.size[::-1]],
    )[0]
    boxes = res["boxes"]
    scores = res["scores"]
    if hasattr(boxes, "cpu"):
        boxes = boxes.cpu().tolist()
        scores = scores.cpu().tolist()
    else:
        boxes = list(boxes)
        scores = list(scores)
    top = max(scores) if scores else None
    annotate_and_save(img, boxes, scores, "omdet_turbo", f"{label}_{category}", name)
    record("omdet_turbo", "text", label, category, name, top, len(scores), ms, threshold)
    record_detections("omdet_turbo", "text", label, category, name,
                      path, boxes, scores, threshold)
    print("omdet_turbo", label, category, name, top, len(scores), f"{ms:.0f}ms")

del om, om_proc
free_memory()

In [ ]:
# Aggregate: write detections.jsonl + summary.json + heatmaps
summary_json = OUTDIR / "ovod_summary.json"
detections_jsonl = OUTDIR / "ovod_detections.jsonl"
n_det_heatmap = OUTDIR / "ovod_n_detections_heatmap.png"
top_score_heatmap = OUTDIR / "ovod_top_score_heatmap.png"

# Clean existing OVOD heatmap PNGs so stale figures don't linger between runs
for old in OUTDIR.glob("ovod_*heatmap*.png"):
    old.unlink()
legacy = OUTDIR / "ovod_heatmap.png"
if legacy.exists():
    legacy.unlink()

with open(summary_json, "w") as f:
    json.dump(results, f, indent=2)

with open(detections_jsonl, "w") as f:
    for d in detections:
        f.write(json.dumps(d) + "\n")

# Build per-model order from the order they appeared in `results`
buckets = ["pos_easy", "pos_hard", "neg_easy", "neg_hard"]
models_in_order = []
seen = set()
for r in results:
    if r["model"] not in seen:
        seen.add(r["model"])
        models_in_order.append(r["model"])


def grid_for(metric):
    """Return a (model x bucket) grid of mean values for the given metric."""
    g = np.full((len(models_in_order), len(buckets)), np.nan)
    for i, model_name in enumerate(models_in_order):
        for j, bucket in enumerate(buckets):
            label, category = bucket.split("_")
            rows = [r for r in results if r["model"] == model_name
                    and r["label"] == label and r["category"] == category]
            vals = [r[metric] for r in rows if r[metric] is not None]
            if vals:
                g[i, j] = float(np.mean(vals))
    return g


def render_heatmap(grid, title, cbar_label, path, cmap="viridis", fmt="{:.2f}"):
    fig, ax = plt.subplots(figsize=(8, max(3, 0.6 * len(models_in_order))))
    im = ax.imshow(grid, aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(buckets)))
    ax.set_xticklabels(buckets)
    ax.set_yticks(range(len(models_in_order)))
    ax.set_yticklabels(models_in_order)
    ax.set_title(title)
    avg = np.nanmean(grid) if np.any(~np.isnan(grid)) else 0.0
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            if not np.isnan(grid[i, j]):
                ax.text(j, i, fmt.format(grid[i, j]),
                        ha="center", va="center",
                        color="white" if grid[i, j] > avg else "black",
                        fontsize=9)
    fig.colorbar(im, ax=ax, label=cbar_label)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)


render_heatmap(
    grid_for("n_detections"),
    "OVOD: mean n_detections per (model, category)",
    "mean n_detections",
    n_det_heatmap,
    fmt="{:.1f}",
)
render_heatmap(
    grid_for("top_score"),
    "OVOD: mean top_score per (model, category)",
    "mean top_score",
    top_score_heatmap,
    fmt="{:.2f}",
)

# Note: top_score is not directly comparable across detectors (different
# calibration: OWLv2 image-guided saturates near 1.0; YOLO-World is 0-1
# calibrated; SAM 3 sits on its own scale). Read the top_score heatmap as
# "is this detector firing harder on positives than negatives within its
# own scale," not as cross-model rankings.

print("wrote", summary_json)
print("wrote", detections_jsonl, "detections:", len(detections))
print("wrote", n_det_heatmap)
print("wrote", top_score_heatmap)